# Oruka Retrieval Bug: Diagnosis Reproduction + Fix Test

This is the companion notebook to `oruka_retrieval_debug_log.md`, and it exists so that the full diagnostic chain we worked through by hand across a long back-and-forth session can actually be re-run from scratch, rather than living only as a written record of what happened. Part 1 walks through every diagnostic step that was originally done manually, turning each one into a real, runnable cell, so you can regenerate the exact evidence that led to the diagnosis without having to remember or retype any of it. Part 2 takes the most promising candidate fix that came out of that diagnosis, a lexical phrase-overlap boost layered on top of the existing vector search, and tests it directly against the specific case that motivated it. Part 3 checks that fix against two questions we already know work well, to make sure a fix for one case doesn't quietly introduce a regression somewhere else.

**The bug, restated plainly:** asking "What three negative claims about African philosophy was Oruka trying to counter?" consistently retrieves a wrong-but-adjacent section of the same article instead of the section that actually states the three claims. The root cause, confirmed step by step in the diagnostic chain below, turned out to be neither a parsing failure nor a stale index, but a genuine embedding-similarity gap: the correct chunk, `Oruka's Project | chunk 1`, ranks 22nd by pure vector similarity with a score of 0.5682, well behind the wrong chunk's score of 0.7989, and behind a whole field of moderately-scoring chunks pulled in from other, thematically-adjacent articles. Two separate things compound to cause this: the correct chunk's content is diluted by an unrelated passage of quote analysis sitting alongside the actual answer, and a cluster of similarly-themed but substantively different articles crowds the ranking further.


In [1]:
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
#print("Working directory set to:", os.getcwd())

Before doing anything else, it's worth explicitly confirming that both the corpus file and the eval test files this notebook depends on are actually where they're expected to be, `SEP/data/SEP.parquet` and `SEP/tests/`, rather than assuming the working-directory fix above was sufficient on its own. This has been a recurring source of confusing errors throughout this project, so a direct check up front is cheaper than debugging a `FileNotFoundError` three cells in.


In [2]:
expected_paths = {
    "SEP/data/SEP.parquet": os.path.join(PROJECT_ROOT, "data", "SEP.parquet"),
    "SEP/tests/": os.path.join(PROJECT_ROOT, "tests"),
    "SEP/tests/eval_systematic.csv": os.path.join(PROJECT_ROOT, "tests", "eval_systematic.csv"),
}

# for label, path in expected_paths.items():
#     status = "FOUND" if os.path.exists(path) else "MISSING"
#     print(f"{status}: {label}  ->  {path}")


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from config import settings
from embeddings import get_embedder
from vectorstore import VectorDB
from section_parser import split_into_sections
from chunker import TextSplitter

embed = get_embedder()
vector_db = VectorDB()
index = vector_db.connect_to_index(settings.PINECONE_INDEX_NAME)

QUESTION = "What three negative claims about African philosophy was Oruka trying to counter?"
TARGET_SECTION = "Oruka\u2019s Project"  # curly apostrophe, matches SEP's actual text
TARGET_CHUNK_NUM = 1


c:\Users\user\anaconda3\envs\deepanalytic\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1: Reproducing the diagnosis

### 1a. Confirming the section parses cleanly, which rules out a TOC-parsing failure as the cause

The first thing worth ruling out, before suspecting anything about embeddings or retrieval, is whether the article's table of contents was even parsed correctly into distinct sections in the first place. If the TOC parsing had silently failed or fallen back to a regex guess, that alone could explain a lot of downstream weirdness, so this check needs to happen before anything else.


In [4]:
df = pd.read_parquet("data/SEP.parquet")
row = df[df["Title"] == "African Sage Philosophy"].iloc[0]

sections, parsed_how = split_into_sections(row["TOC"], row["Text"])
print("parsed_how:", parsed_how)
for title, text in sections:
    print(" -", repr(title))


parsed_how: toc_match
 - 'Oruka’s Project'
 - 'The African Sage Tradition and Eurocentric Bias'
 - 'Literacy and The Oral Tradition in Sage Philosophy'
 - 'Ethnophilosophy, Unanimity and African Critical Thought'
 - 'What counts as Sage Philosophy?'


### 1b. Confirming the target chunk's actual content, which rules out a chunking-boundary failure as the cause

The next thing worth checking directly, rather than assuming, is what the actual text of the chunk containing the three-claims sentence looks like once it's been split by the current chunking logic. It's entirely possible for a section to parse correctly at the TOC level and still end up chunked in a way that buries or fragments the specific sentence that matters, so this step pulls the real chunk text rather than reasoning about it in the abstract.


In [5]:
splitter = TextSplitter(chunk_size=400, chunk_overlap=20)

for title, text in sections:
    if "Oruka" in title:
        chunks = splitter.split_text(text)
        print(f"{len(chunks)} chunks for this section")
        print()
        print(f"--- chunk {TARGET_CHUNK_NUM} (the target) ---")
        print(chunks[TARGET_CHUNK_NUM])


2 chunks for this section

--- chunk 1 (the target) ---
So that they may eat… so that they may get empty prestige. They
want to profit fraudulently. (Sage Philosophy, p. 111)


From these examples some of the distinguishing characteristics of Sage
Philosophy can be gleaned. First, they display the deeply personal
nature of the ideas, or opinions, that the sages expressed in response
to the questions. Akoko’s insight derives from his individual
reflections on the practice of communalism to consider its
justification. Second, they provide evidence of abstract thought about
philosophic topics. Chaungo considers what it means for a proposition
to be true and expresses what turns out to be a correspondence theory
of truth — according to which the proposition “This is a
bottle” is true if the object it refers to, this thing, is
indeed a bottle. By pointing out that some people choose deliberately
to be untruthful for unjust gain he also addresses the moral aspects
of truth.

Oruka’s survey o

### 1c. Confirming the chunk is actually live in the production index, which rules out staleness as the cause

Given how much the chunking code changed over the course of this project, it was genuinely worth asking whether the production Pinecone index still reflected an older version of that code, rather than what the current codebase would produce if run fresh. This step queries the live index directly by metadata filter, not by similarity, specifically to see whether the target chunk actually exists as a stored vector, independent of whether it ranks well for any particular question.


In [6]:
results = index.query(
    vector=[0.0] * 1536,
    top_k=50,
    filter={"title": {"$eq": "African Sage Philosophy"}},
    include_metadata=True,
)

for match in results["matches"]:
    meta = match["metadata"]
    if "Oruka" in meta.get("section", ""):
        print(meta.get("section"), "| chunk", meta.get("chunk"))


Oruka’s Project | chunk 1.0
Oruka’s Project | chunk 0.0


### 1d. The actual rank-position check, which is where the real diagnosis came from

This is the step that actually settled the question. Rather than just checking whether the target chunk appears in the usual top-k retrieved set, this runs a full similarity search with a much larger k of 50, so we can see exactly where the target chunk lands in the full ranked order and, just as importantly, exactly what is outranking it and by how much. That second part turned out to matter as much as the rank number itself.


In [7]:
query_vector = embed.embed_query(QUESTION)
search_results = index.query(vector=query_vector, top_k=50, include_metadata=True)

target_rank = None
for rank, match in enumerate(search_results["matches"], 1):
    meta = match["metadata"]
    is_target = "Oruka" in meta.get("section", "") and meta.get("chunk") == TARGET_CHUNK_NUM
    if is_target:
        target_rank = rank
    marker = " <-- TARGET" if is_target else ""
    print(f"{rank}. score={match['score']:.4f}  {meta.get('title')} | {meta.get('section')} | chunk {meta.get('chunk')}{marker}")

print(f"\nTarget chunk rank: {target_rank}")


1. score=0.7988  African Sage Philosophy | Ethnophilosophy, Unanimity and African Critical Thought | chunk 1.0
2. score=0.7104  African Sage Philosophy | Literacy and The Oral Tradition in Sage Philosophy | chunk 1.0
3. score=0.6729  African Sage Philosophy | Ethnophilosophy, Unanimity and African Critical Thought | chunk 2.0
4. score=0.6719  African Sage Philosophy | Literacy and The Oral Tradition in Sage Philosophy | chunk 2.0
5. score=0.6544  African Sage Philosophy | What counts as Sage Philosophy? | chunk 8.0
6. score=0.6454  African Sage Philosophy | The African Sage Tradition and Eurocentric Bias | chunk 0.0
7. score=0.6378  African Sage Philosophy | What counts as Sage Philosophy? | chunk 7.0
8. score=0.6326  African Sage Philosophy | Literacy and The Oral Tradition in Sage Philosophy | chunk 4.0
9. score=0.6195  Africana Philosophy | Africana Philosophy: Continental Africa | chunk 9.0
10. score=0.6169  Africana Philosophy | Africana Philosophy: Continental Africa | chunk 16.0

## Part 2: Testing a candidate fix, a lexical phrase-overlap boost

The working hypothesis behind this fix is fairly specific: the correct chunk's actual phrasing, "aimed to counter three negative claims," sits close to verbatim overlap with the question's own wording, "three negative claims," while the wrong chunk's phrasing, "the third negative claim," is an ordinal reference to a single item rather than the same multi-word phrase the question uses. Pure semantic similarity doesn't appear to be picking up on that distinction, since it's scoring the wrong chunk far higher despite the weaker phrase match, so the idea here is to add a  lexical signal on top of the existing vector similarity and see whether that shifts things in the right direction.

Rather than standing up a separate BM25 search engine just to test this idea, this uses TF-IDF cosine similarity as a lightweight but legitimate lexical-overlap signal, computed directly against the same 50 candidates already retrieved above, so the comparison stays apples to apples.


In [11]:
candidate_texts = [m["metadata"].get("text", "") for m in search_results["matches"]]
candidate_meta = [m["metadata"] for m in search_results["matches"]]
vector_scores = [m["score"] for m in search_results["matches"]]

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform([QUESTION] + candidate_texts)
lexical_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()

# Blend: weighted sum of vector similarity and lexical similarity.
# This starting weight is a genuine guess, not a tuned value -- see the
# closing note at the end of this notebook before trusting it too far.
LEXICAL_WEIGHT = 0.5
blended_scores = [
    (1 - LEXICAL_WEIGHT) * v + LEXICAL_WEIGHT * l
    for v, l in zip(vector_scores, lexical_scores)
]

reranked = sorted(
    zip(blended_scores, vector_scores, lexical_scores, candidate_meta),
    key=lambda x: x[0],
    reverse=True,
)

new_target_rank = None
for rank, (blended, vscore, lscore, meta) in enumerate(reranked, 1):
    is_target = "Oruka" in meta.get("section", "") and meta.get("chunk") == TARGET_CHUNK_NUM
    if is_target:
        new_target_rank = rank
    marker = " <-- TARGET" if is_target else ""
    if rank <= 15 or is_target:
        print(f"{rank}. blended={blended:.4f} (vec={vscore:.4f}, lex={lscore:.4f})  {meta.get('title')} | {meta.get('section')} | chunk {meta.get('chunk')}{marker}")

print(f"\nTarget chunk rank -- before: {target_rank}, after lexical blend: {new_target_rank}")


1. blended=0.4554 (vec=0.7988, lex=0.1119)  African Sage Philosophy | Ethnophilosophy, Unanimity and African Critical Thought | chunk 1.0
2. blended=0.4098 (vec=0.7104, lex=0.1092)  African Sage Philosophy | Literacy and The Oral Tradition in Sage Philosophy | chunk 1.0
3. blended=0.3729 (vec=0.6719, lex=0.0738)  African Sage Philosophy | Literacy and The Oral Tradition in Sage Philosophy | chunk 2.0
4. blended=0.3721 (vec=0.6729, lex=0.0713)  African Sage Philosophy | Ethnophilosophy, Unanimity and African Critical Thought | chunk 2.0
5. blended=0.3709 (vec=0.6009, lex=0.1409)  African Sage Philosophy | Literacy and The Oral Tradition in Sage Philosophy | chunk 0.0
6. blended=0.3668 (vec=0.6326, lex=0.1011)  African Sage Philosophy | Literacy and The Oral Tradition in Sage Philosophy | chunk 4.0
7. blended=0.3658 (vec=0.6544, lex=0.0772)  African Sage Philosophy | What counts as Sage Philosophy? | chunk 8.0
8. blended=0.3610 (vec=0.5682, lex=0.1537)  African Sage Philosophy | Oruka’s 

## Part 3: Checking for regressions on questions that already worked

A fix that helps this one case is only worth adopting if it doesn't quietly break something that was already working correctly under plain vector search. This section pulls two known-good questions directly from `SEP/tests/eval_systematic.csv`, the same file the main naive-vs-rerank eval was built from, rather than hardcoding fresh questions here, so the regression check stays grounded in the project's existing, already-validated test set instead of drifting into ad hoc examples.


In [12]:
eval_df = pd.read_csv("tests/eval_systematic.csv")

# Pull two known-good, already-validated questions from the real eval set,
# one from Ancient Political Philosophy and one from Animalism, rather than
# writing new ad hoc questions just for this regression check.
regression_questions = eval_df[
    eval_df["Question"].str.contains("Cicero", case=False, na=False)
    | eval_df["Question"].str.contains("thinking animal", case=False, na=False)
]["Question"].drop_duplicates().tolist()

print("Regression questions pulled from tests/eval_systematic.csv:")
for q in regression_questions:
    print(" -", q)


Regression questions pulled from tests/eval_systematic.csv:
 - What is the thinking animal argument?
 - Why does Cicero say Rome under the Republic satisfies the definition of a res publica?


In [13]:
def check_regression(question):
    qv = embed.embed_query(question)
    res = index.query(vector=qv, top_k=20, include_metadata=True)

    texts = [m["metadata"].get("text", "") for m in res["matches"]]
    meta = [m["metadata"] for m in res["matches"]]
    vscores = [m["score"] for m in res["matches"]]

    tfidf = vectorizer.fit_transform([question] + texts)
    lscores = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
    blended = [(1 - LEXICAL_WEIGHT) * v + LEXICAL_WEIGHT * l for v, l in zip(vscores, lscores)]

    top_before = meta[vscores.index(max(vscores))]
    reranked_pairs = sorted(zip(blended, meta), key=lambda x: x[0], reverse=True)
    top_after = reranked_pairs[0][1]

    print(f"Q: {question}")
    print(f"  top result BEFORE (pure vector): {top_before.get('title')} | {top_before.get('section')}")
    print(f"  top result AFTER (blended):      {top_after.get('title')} | {top_after.get('section')}")
    print()


for q in regression_questions:
    check_regression(q)


Q: What is the thinking animal argument?
  top result BEFORE (pure vector): Animalism | Arguments for and Objections to Animalism
  top result AFTER (blended):      Animalism | Arguments for and Objections to Animalism

Q: Why does Cicero say Rome under the Republic satisfies the definition of a res publica?
  top result BEFORE (pure vector): Ancient Political Philosophy | The Roman Republic and Cicero
  top result AFTER (blended):      Ancient Political Philosophy | The Roman Republic and Cicero



## Where this leaves things

This has now actually been run, and the results were folded back into `oruka_retrieval_debug_log.md`: at `LEXICAL_WEIGHT = 0.3`, the target chunk's rank moved from 22 to 13, with its lexical score coming out as the highest in its neighborhood, confirming the phrase-overlap theory was correct in direction. Pushing the weight to 0.5 moved the target chunk further, to rank 8, comfortably inside a realistic retrieval window, and the regression check against two known-good control questions from `tests/eval_systematic.csv` came back identical before and after the blend on both, no cost observed on cases that already worked.

That result is very promising, but it is not the same thing as this being production-ready. Three things remain open before this becomes anything more than a tested experiment: 
- whether a weight tuned against three questions actually generalizes to a wider set rather than happening to work well on exactly the cases checked here, 
- whether this should become the default scoring behavior for every query the pipeline runs or stay an optional mode reserved for cases that look like they need it, and 
- whether rank 8 is a stable result or one that a slightly different phrasing of the same question would lose. None of this is wired into `rag_pipeline.py` yet, the real query methods a user would actually hit still do plain vector search, unchanged. See `oruka_retrieval_debug_log.md` for the full record of what's been tested and what's still pending.